# Notebook 01: Original Model Training (`01_train_full.ipynb`)

### Objectives & Workflow:
1. **Deterministic Dataset Partitioning**: Create and persist deterministic `D` (42,500), `D_V` (7,500), `D_R` (38,250), `D_F` (4,250), and `D_T` (10,000) splits according to `data_instruction.md`.
2. **Original Model Training ($	heta_o$)**: Train model (`resnet18` / `vit11m`) on full training set $D = D_R \cup D_F$ using hyperparameter configurations matching `model_instruction.md`.
3. **Model Selection**: Select best checkpoint strictly on validation set $D_V$.
4. **Evaluation**: Compute 3-tier metrics (Output, LP, NCC) on validation (for selection) and test (final reporting only).
5. **Artifacts**: Save checkpoint, configuration, training history, and split files across model seeds `[0, 1, ..., 9]`.

In [ ]:
import os
import copy
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

from experiment_utils import (
    build_classifier,
    create_and_persist_splits,
    get_dataset_transforms,
    IndexedDataset,
    compute_output_accuracy,
    evaluate_three_tier_metrics,
    MODEL_CONFIGS
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. Setup Configurations & Deterministic Partitions (from `model_instruction.md`)

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Fixed split seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

# Load hyperparameters directly matching model_instruction.md
cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
BATCH_SIZE = cfg["batch_size"]    # 256
EPOCHS = cfg["num_epochs"]        # 182 for cifar10/100, 50 for mnist/fashion/utkface
LR = cfg["lr"]                    # 0.1
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)
WEIGHT_DECAY = 5e-4
MOMENTUM = 0.9

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)
print(f"Config: Dataset={DATASET_NAME}, Model={MODEL_NAME}, Epochs={EPOCHS}, BatchSize={BATCH_SIZE}, LR={LR}, Seeds={MODEL_SEEDS}")

## 2. Training Loop for Original Model

In [ ]:
train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
raw_train = datasets.CIFAR10(root="./data", train=True, download=True)
raw_test = datasets.CIFAR10(root="./data", train=False, download=True)

# 1. Create and persist split using fixed SPLIT_SEED = 123
split_info = create_and_persist_splits(
    dataset_name=DATASET_NAME,
    root="./data",
    seed=SPLIT_SEED,
    save_dir="./artifacts/splits"
)

# 2. Build Datasets and DataLoaders
train_set = IndexedDataset(Subset(raw_train, split_info["train_indices"]), train_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
train_r_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
train_f_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
train_r_loader = DataLoader(train_r_set, batch_size=BATCH_SIZE, shuffle=False)
train_f_loader = DataLoader(train_f_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

for seed in MODEL_SEEDS:
    print(f"\n==================================================")
    print(f"--- Training Original Model | Model: {MODEL_NAME} | Seed: {seed} ---")
    print(f"==================================================")
    
    # 3. Model & Optimizer Setup
    torch.manual_seed(seed)
    model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    best_val_acc = 0.0
    best_state = None
    history = []
    
    # 4. Training Loop
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
        scheduler.step()
        
        # Validation evaluation strictly for model selection
        val_acc = compute_output_accuracy(model, val_loader, device=device)
        history.append({"epoch": epoch, "train_loss": total_loss / len(train_set), "val_acc": val_acc})
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            
        if epoch % 10 == 0 or epoch == EPOCHS:
            print(f"Epoch [{epoch:03d}/{EPOCHS:03d}] - Loss: {total_loss / len(train_set):.4f} - Val Acc: {val_acc:.4f} (Best: {best_val_acc:.4f})")
    
    # 5. Restore best checkpoint
    model.load_state_dict(best_state)
    ckpt_path = f"./artifacts/checkpoints/original_{MODEL_NAME}_seed_{seed}.pt"
    torch.save(best_state, ckpt_path)
    print(f"Saved best checkpoint to: {ckpt_path}")
    
    # 6. Evaluate 3-tier metrics on Validation and Test
    val_metrics = evaluate_three_tier_metrics(
        model=model, train_loader=train_loader, retain_eval_loader=train_r_loader, forget_eval_loader=train_f_loader, num_classes=NUM_CLASSES, seed=seed, device=device
    )
    test_metrics = evaluate_three_tier_metrics(
        model=model, train_loader=train_loader, retain_eval_loader=test_loader, forget_eval_loader=train_f_loader, num_classes=NUM_CLASSES, seed=seed, device=device
    )
    
    results = {
        "model_type": "original",
        "model_name": MODEL_NAME,
        "seed": seed,
        "split_seed": SPLIT_SEED,
        "validation_metrics": val_metrics,
        "test_metrics": test_metrics,
        "training_history": history
    }
    
    res_path = f"./artifacts/metrics/original_{MODEL_NAME}_seed_{seed}.json"
    with open(res_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Test Output Acc Retain: {test_metrics['output_retain']:.4f} | Forget: {test_metrics['output_forget']:.4f}")
    print(f"Test LP Acc Retain: {test_metrics['lp_retain']:.4f} | Forget: {test_metrics['lp_forget']:.4f}")
    print(f"Test NCC Acc Retain: {test_metrics['ncc_retain']:.4f} | Forget: {test_metrics['ncc_forget']:.4f}")
    print(f"Saved metrics summary to: {res_path}")